# INTEGRA — Google Colab Environment Setup
### Automated Lecture Video Summarization

> **Use this notebook if your local machine does not have an NVIDIA GPU.**
>
> **FIRST: Runtime → Change runtime type → T4 GPU → Save**

---

Run each cell from top to bottom. Do not skip any cell.

In [ ]:
# ─── Cell 1: Verify GPU ────────────────────────────────────────────────────
# MUST show CUDA: True before continuing.
# If it shows False → Runtime → Change runtime type → T4 GPU → Save → Re-run.

import torch
print('='*55)
print('  INTEGRA GPU Check')
print('='*55)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {vram:.1f} GB')
    print('\n  ✅ GPU ready — proceed to Cell 2')
else:
    print('\n  ❌ No GPU detected!')
    print('  → Runtime → Change runtime type → T4 GPU → Save → Re-run this cell')

In [ ]:
# ─── Cell 2: Mount Google Drive ────────────────────────────────────────────
# This saves your work between Colab sessions.
# After mounting, your Drive is available at /content/drive/MyDrive/

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/INTEGRA/outputs',     exist_ok=True)
os.makedirs('/content/drive/MyDrive/INTEGRA/models',      exist_ok=True)
os.makedirs('/content/drive/MyDrive/INTEGRA/annotations', exist_ok=True)
print('\n  ✅ Drive mounted. Your persistent storage is at /content/drive/MyDrive/INTEGRA/')

In [ ]:
# ─── Cell 3: Clone Repository ──────────────────────────────────────────────
# Change MODULE_BRANCH to your module: module-1, module-2, module-3, module-4

MODULE_BRANCH = 'module-1'   # <── CHANGE THIS to your module branch

import os
if not os.path.exists('/content/lecture-video-summarizer'):
    !git clone https://github.com/rashmiJayawardhana/lecture-video-summarizer.git

%cd /content/lecture-video-summarizer
!git checkout {MODULE_BRANCH}
!git pull origin {MODULE_BRANCH}
print(f'\n  ✅ On branch: {MODULE_BRANCH}')
!git log --oneline -3

In [ ]:
# ─── Cell 4: Install Dependencies ──────────────────────────────────────────
# Colab already has PyTorch with CUDA — we install the rest.

%cd /content/lecture-video-summarizer
!pip install -r requirements.txt --quiet

# Verify key packages
import importlib
packages = {
    'cv2':          'opencv-python',
    'transformers': 'transformers',
    'whisper':      'openai-whisper',
    'moviepy':      'moviepy',
    'rouge_score':  'rouge-score',
    'jiwer':        'jiwer',
}
print('\nPackage verification:')
for mod, pkg in packages.items():
    try:
        m = importlib.import_module(mod)
        print(f'  [PASS] {pkg} ({getattr(m, "__version__", "ok")})')
    except ImportError:
        print(f'  [FAIL] {pkg} — run: !pip install {pkg}')

In [ ]:
# ─── Cell 5: Verify FFmpeg ─────────────────────────────────────────────────
!ffmpeg -version 2>&1 | head -1
print('  ✅ FFmpeg is pre-installed on Colab')

In [ ]:
# ─── Cell 6: Set Up Environment Variables ──────────────────────────────────
# Add your API keys and paths here.
# These are NOT committed to GitHub — they only exist in this session.

import os

# Only needed for Module 4 Pipeline B (Lathisana)
os.environ['OPENAI_API_KEY'] = ''   # Paste your key here if you are Module 4

os.environ['DATA_DIR']    = '/content/lecture-video-summarizer/data'
os.environ['OUTPUT_DIR']  = '/content/lecture-video-summarizer/outputs'
os.environ['MODELS_DIR']  = '/content/drive/MyDrive/INTEGRA/models'  # Use Drive for persistence

print('  ✅ Environment variables set')

In [ ]:
# ─── Cell 7: Module-Specific Verification ──────────────────────────────────
# Uncomment and run the block for YOUR module.

MODULE = 1   # <── CHANGE to your module number: 1, 2, 3, or 4

if MODULE == 1:
    # Rashmi — ResNet-50 + BiLSTM
    import torchvision.models as models
    resnet = models.resnet50(weights='IMAGENET1K_V1')
    params = sum(p.numel() for p in resnet.parameters())
    print(f'  ✅ ResNet-50 loaded: {params:,} parameters')
    import torch
    dummy = torch.randn(1, 30, 3, 224, 224).cuda()
    print(f'  ✅ Dummy tensor on GPU: {dummy.shape}')

elif MODULE == 2:
    # Ravindu — Whisper + BERT
    import whisper
    model = whisper.load_model('tiny')
    print('  ✅ Whisper tiny loaded')
    # To cache large-v3 (do this once, takes ~5 min):
    # model = whisper.load_model('large-v3')
    # print('  ✅ Whisper large-v3 cached')

elif MODULE == 3:
    # Fazly — ViT + TrOCR + OpenCV
    from transformers import ViTForImageClassification
    vit = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
    print('  ✅ ViT-base loaded')
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    proc  = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
    trocr = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
    print('  ✅ TrOCR loaded')

elif MODULE == 4:
    # Lathisana — MoviePy + edge-tts + GPT-4o
    from moviepy.editor import VideoFileClip
    print('  ✅ MoviePy imported')
    !python -m edge_tts --voice en-US-AriaNeural --text "INTEGRA test." --write-media /tmp/test.mp3
    import os
    print('  ✅ edge-tts OK' if os.path.exists('/tmp/test.mp3') else '  ❌ edge-tts failed')

print(f'\n  ✅ Module {MODULE} environment ready!')

In [ ]:
# ─── Cell 8: Save Progress to Drive & Push to GitHub ──────────────────────
# Run this at the END of every session to avoid losing work.

import shutil, os

# Copy outputs to Drive (persistent)
for folder in ['outputs', 'data/annotations']:
    src = f'/content/lecture-video-summarizer/{folder}'
    dst = f'/content/drive/MyDrive/INTEGRA/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  Copied {folder} → Drive')

# Push code changes to GitHub
%cd /content/lecture-video-summarizer
!git config user.email "your-email@example.com"  # Replace with your email
!git config user.name  "Your Name"               # Replace with your name
!git add .
!git status
# Uncomment below to commit and push:
# commit_msg = "Week 1: initial module work"
# !git commit -m "{commit_msg}" && git push

print('\n  ✅ Session saved. Uncomment the commit lines above to push to GitHub.')

---

## Notes for Colab Users

| Issue | Fix |
|-------|-----|
| Session disconnects after ~12 hours | Save checkpoints to Drive in Cell 8 |
| Large models download slowly | Run model download once, save to Drive |
| `CUDA out of memory` | Reduce `batch_size` in your config YAML |
| Whisper large-v3 OOM on T4 | Use `whisper.load_model('medium')` first |
| Push to GitHub fails | Use a Personal Access Token instead of password |

**Always run Cell 8 before your session ends.**

*Setup guide: `docs/SETUP_GUIDE.md` | Questions: raise in weekly call.*